# Task 1 - Customer Support Triage Pipeline
 
## Objective
 
Build a multi-agent workflow using LangGraph that:
 
- Classifies customer requests
- Retrieves relevant Knowledge Base (KB) articles
- Generates a professional customer response
 
---


## Workflow
 
Customer Query
        │
        ▼
Classifier Agent
        │
        ▼
Retrieval Agent
        │
        ▼
Composer Agent
        │
        ▼
Final Response

In [6]:
# ======================================
# Step 1 : Import Required Libraries
# ======================================
 
import os
from dotenv import load_dotenv
from typing import TypedDict
 
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic

In [7]:
# ======================================
# Step 2 : Load Environment Variables
# ======================================
 
load_dotenv(override=True)
 
llm = ChatAnthropic(
    model=os.getenv("LLM_MODEL"),
    base_url=os.getenv("LLM_ENDPOINT"),
    temperature=0
)
 
print("LLM Loaded Successfully")

LLM Loaded Successfully


In [ ]:


# ====================================== 
#Define the LangGraph State.
 
#This state carries data from one agent to another.
# ======================================



# ======================================
# Step 3 : Define Graph State
# ======================================
 
class CustomerState(TypedDict):
    query: str
    category: str
    priority: str
    kb_result: str
    response: str

In [9]:

# ======================================
# Step 4 : Knowledge Base
# ======================================
 
knowledge_base = {
 
    "billing":
    """
Billing Issues
 
• Payments usually reflect within 24 hours.
• Refunds take 5-7 business days.
• Download invoices from Billing Settings.
""",
 
    "technical":
    """
Technical Support
 
• Restart the application.
• Clear browser cache.
• Update to the latest version.
• Contact support if the issue continues.
""",
 
    "account":
    """
Account Access
 
• Reset password using Forgot Password.
• Unlock account after identity verification.
• Enable Two-Factor Authentication.
""",
 
    "general":
    """
General Support
 
• Visit the Help Center.
• Contact customer support.
"""
}
 
print("Knowledge Base Ready")

Knowledge Base Ready


## Step 5
 
Classifier Agent
 
Responsibilities:
 
- Identify customer intent
- Assign priority
- Store the result in the graph state

In [12]:
# ======================================
# Step 5 : Classifier Agent
# ======================================
 
def classifier_agent(state: CustomerState):
 
    prompt = f"""
You are a Customer Support Classifier.
 
Classify the customer query.
 
Possible Categories:
- billing
- technical
- account
- general
 
Priority Levels:
- High
- Medium
- Low
 
Return ONLY in this format:
 
Category: billing
Priority: High
 
Customer Query:
{state["query"]}
"""
 
    result = llm.invoke(prompt).content
 
    print("Classifier Output")
    print(result)
 
    category = "general"
    priority = "Low"
 
    for line in result.split("\n"):
 
        if line.lower().startswith("category"):
            category = line.split(":")[1].strip().lower()
 
        if line.lower().startswith("priority"):
            priority = line.split(":")[1].strip()
 
    state["category"] = category
    state["priority"] = priority
 
    return state

In [13]:
#Test Only the Classifier
sample = {
    "query": "I cannot login to my account."
}
 
classifier_agent(sample)

Classifier Output
Category: account
Priority: High


{'query': 'I cannot login to my account.',
 'category': 'account',
 'priority': 'High'}


## Step 6
 
### Retrieval Agent
 
This agent searches the Knowledge Base using the category predicted by the Classifier Agent.
 
The retrieved knowledge will be passed to the Composer Agent.

In [14]:

# ======================================
# Step 6 : Retrieval Agent
# ======================================
 
def retrieval_agent(state: CustomerState):
 
    category = state["category"]
 
    kb_result = knowledge_base.get(category, knowledge_base["general"])
 
    print("Retrieved Knowledge Base")
    print(kb_result)
 
    state["kb_result"] = kb_result
 
    return state

## Step 7
 
### Composer Agent
 
This agent combines:
 
- Customer Query
- Category
- Priority
- Retrieved Knowledge Base
 
to generate a professional customer-facing response.

In [15]:

# ======================================
# Step 7 : Composer Agent
# ======================================
 
def composer_agent(state: CustomerState):
 
    prompt = f"""
You are a professional customer support executive.
 
Customer Query:
{state["query"]}
 
Category:
{state["category"]}
 
Priority:
{state["priority"]}
 
Knowledge Base:
{state["kb_result"]}
 
Generate a polite and professional customer response.
"""
 
    answer = llm.invoke(prompt)
 
    print("Final Response Generated")
 
    state["response"] = answer.content
 
    return state

 
### Build the LangGraph
 
Workflow:
 
START
 
↓
 
Classifier Agent
 
↓
 
Retrieval Agent
 
↓
 
Composer Agent
 
↓
 
END

In [16]:

# ======================================
# Step 8 : Create Graph Builder
# ======================================
 
builder = StateGraph(CustomerState)

# ======================================
# Step 9 : Add Nodes
# ======================================
 
builder.add_node("classifier", classifier_agent)
 
builder.add_node("retrieval", retrieval_agent)
 
builder.add_node("composer", composer_agent)

# ======================================
# Step 10 : Connect Graph
# ======================================
 
builder.add_edge(START, "classifier")
 
builder.add_edge("classifier", "retrieval")
 
builder.add_edge("retrieval", "composer")
 
builder.add_edge("composer", END)

In [17]:
# ======================================
# Step 11 : Compile Graph
# ======================================
 
graph = builder.compile()
 
print("Customer Support Graph Created Successfully")

Customer Support Graph Created Successfully


# Step 12
 
## Test the Customer Support Pipeline
 
We will test the workflow using five different customer queries.
 
For each query we will display:
 
- Customer Query
- Classified Category
- Priority
- Retrieved Knowledge Base
- Final Customer Response

In [19]:

# ======================================
# Test Case 1 : Billing Issue
# ======================================
 
result = graph.invoke({
    "query": "I was charged twice for my subscription."
})
 
print("=" * 60)
print("Customer Query")
print(result["query"])
 
print("\nCategory")
print(result["category"])
 
print("\nPriority")
print(result["priority"])
 
print("\nKnowledge Base")
print(result["kb_result"])
 
print("\nFinal Response")
print(result["response"])




Classifier Output
Category: billing
Priority: High
Retrieved Knowledge Base

Billing Issues

• Payments usually reflect within 24 hours.
• Refunds take 5-7 business days.
• Download invoices from Billing Settings.

Final Response Generated
Customer Query
I was charged twice for my subscription.

Category
billing

Priority
High

Knowledge Base

Billing Issues

• Payments usually reflect within 24 hours.
• Refunds take 5-7 business days.
• Download invoices from Billing Settings.


Final Response
# Customer Support Response

Dear Valued Customer,

Thank you for reaching out, and I sincerely apologize for the inconvenience you've experienced with the duplicate charge on your subscription.

I understand how frustrating this must be, and I want to assure you that we take this matter seriously. Here's what I can help you with:

**Immediate Next Steps:**
1. I will investigate your account to confirm the duplicate charge
2. Once verified, I will process a refund for the erroneous charge immedi

In [21]:
# ======================================
# Test Case 2 : Technical Issue
# ======================================
 
result = graph.invoke({
    "query": "The application crashes whenever I login."
})
 
print("=" * 60)
print("Customer Query")
print(result["query"])
 
print("\nCategory")
print(result["category"])
 
print("\nPriority")
print(result["priority"])
 
print("\nKnowledge Base")
print(result["kb_result"])
 
print("\nFinal Response")
print(result["response"])


Classifier Output
Category: technical
Priority: High
Retrieved Knowledge Base

Technical Support

• Restart the application.
• Clear browser cache.
• Update to the latest version.
• Contact support if the issue continues.

Final Response Generated
Customer Query
The application crashes whenever I login.

Category
technical

Priority
High

Knowledge Base

Technical Support

• Restart the application.
• Clear browser cache.
• Update to the latest version.
• Contact support if the issue continues.


Final Response
# Customer Support Response

Dear Valued Customer,

Thank you for reaching out to us, and I sincerely apologize for the inconvenience you're experiencing with the application crashing during login.

I understand how frustrating this must be, and I'm here to help resolve this issue for you right away. Please try the following troubleshooting steps:

1. **Restart the Application** – Close the application completely and reopen it.

2. **Clear Your Browser Cache** – This often resol

In [22]:
# ======================================
# Test Case 3 : Account Issue
# ======================================
 
result = graph.invoke({
    "query": "I forgot my password and cannot access my account."
})
 
print("=" * 60)
print("Customer Query")
print(result["query"])
 
print("\nCategory")
print(result["category"])
 
print("\nPriority")
print(result["priority"])
 
print("\nKnowledge Base")
print(result["kb_result"])
 
print("\nFinal Response")
print(result["response"])

Classifier Output
Category: account
Priority: High
Retrieved Knowledge Base

Account Access

• Reset password using Forgot Password.
• Unlock account after identity verification.
• Enable Two-Factor Authentication.

Final Response Generated
Customer Query
I forgot my password and cannot access my account.

Category
account

Priority
High

Knowledge Base

Account Access

• Reset password using Forgot Password.
• Unlock account after identity verification.
• Enable Two-Factor Authentication.


Final Response
# Customer Support Response

Dear Valued Customer,

Thank you for reaching out to us, and I sincerely apologize for the inconvenience you're experiencing.

I'm here to help you regain access to your account right away. Here are the steps I recommend:

**Immediate Solution:**
1. Visit our login page and click on **"Forgot Password"**
2. Enter your registered email address
3. Check your email inbox (and spam folder) for a password reset link
4. Follow the instructions in the email to c

In [23]:
# ======================================
# Test Case 4 : General Query
# ======================================
 
result = graph.invoke({
    "query": "Where can I find your customer support email?"
})
 
print("=" * 60)
print("Customer Query")
print(result["query"])
 
print("\nCategory")
print(result["category"])
 
print("\nPriority")
print(result["priority"])
 
print("\nKnowledge Base")
print(result["kb_result"])
 
print("\nFinal Response")
print(result["response"])

Classifier Output
Category: general
Priority: Low
Retrieved Knowledge Base

General Support

• Visit the Help Center.
• Contact customer support.

Final Response Generated
Customer Query
Where can I find your customer support email?

Category
general

Priority
Low

Knowledge Base

General Support

• Visit the Help Center.
• Contact customer support.


Final Response
# Response

Thank you for reaching out to us!

You can find our customer support contact information through our **Help Center**, where we provide multiple ways to get in touch with our team. This includes email support options along with other helpful resources.

If you'd like direct assistance, please visit our Help Center, and you'll be able to locate our customer support email and other contact methods that best suit your needs.

Is there anything specific I can help you with today?

---

*Best regards,*  
*Customer Support Team*


In [24]:
# ======================================
# Test Case 5 : Mixed Issue
# ======================================
 
result = graph.invoke({
    "query": "My payment failed and now I cannot login."
})
 
print("=" * 60)
print("Customer Query")
print(result["query"])
 
print("\nCategory")
print(result["category"])
 
print("\nPriority")
print(result["priority"])
 
print("\nKnowledge Base")
print(result["kb_result"])
 
print("\nFinal Response")
print(result["response"])

Classifier Output
Category: account
Priority: High
Retrieved Knowledge Base

Account Access

• Reset password using Forgot Password.
• Unlock account after identity verification.
• Enable Two-Factor Authentication.

Final Response Generated
Customer Query
My payment failed and now I cannot login.

Category
account

Priority
High

Knowledge Base

Account Access

• Reset password using Forgot Password.
• Unlock account after identity verification.
• Enable Two-Factor Authentication.


Final Response
# Customer Support Response

Dear Valued Customer,

Thank you for reaching out to us, and I sincerely apologize for the inconvenience you're experiencing.

I understand you're unable to login following a failed payment, and I'm here to help resolve this for you right away.

**Here's what we can do:**

1. **Account Access** – Let's first restore your login access:
   - Try using our **Forgot Password** feature to reset your credentials
   - If you continue to experience issues, your account ma